## Run your Llama Stack distribution

Nuke any old distro config files you might have lying around (I find these get in the way whenever I change my `.env` variables): 
```bash
ls ~/.llama/distributions/
rm -r ~/.llama/distributions/<name-of-your-distro>
```

Then, run your llama stack server with:
```bash
dotenv run uv run llama stack run distribution/run.yaml
```




## Setup and Imports


In [5]:
# Install dev packages if not already installed
# !uv pip install -e ".[dev]"

# You will also need:
# uv pip install sentence-transformers

import os

import pandas as pd
import sentence_transformers  # noqa
from datasets import Dataset, load_dataset
from langchain.chat_models import BaseChatModel, init_chat_model
from langchain_community.document_loaders import HuggingFaceDatasetLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from llama_stack_client import LlamaStackClient
from rich.pretty import pprint
from tqdm import tqdm


## Llama Stack Client Setup

- Make sure we have an inference model (model_type='llm')
- Make sure we have an embedding model (model_type='embedding')


In [6]:
# If using the remote provider, you will need ngrok to enable remote access to your Llama Stack server
# Otherwise, the base_url is just http://localhost:8321

client = LlamaStackClient(base_url=os.getenv("KUBEFLOW_LLAMA_STACK_URL"))
available_models = client.models.list()
pprint(available_models)

INFO:httpx:HTTP Request: GET https://2b82-174-95-180-98.ngrok-free.app/v1/models "HTTP/1.1 200 OK"


[
│   Model(
│   │   id='ollama/all-minilm:latest',
│   │   created=1770492896,
│   │   owned_by='llama_stack',
│   │   custom_metadata={
│   │   │   'model_type': 'embedding',
│   │   │   'provider_id': 'ollama',
│   │   │   'provider_resource_id': 'all-minilm:latest',
│   │   │   'embedding_dimension': 384
│   │   },
│   │   object='model'
│   ),
│   Model(
│   │   id='ollama/granite3.3:2b',
│   │   created=1770492896,
│   │   owned_by='llama_stack',
│   │   custom_metadata={'model_type': 'llm', 'provider_id': 'ollama', 'provider_resource_id': 'granite3.3:2b'},
│   │   object='model'
│   ),
│   Model(
│   │   id='ollama/granite4:latest',
│   │   created=1770492896,
│   │   owned_by='llama_stack',
│   │   custom_metadata={'model_type': 'llm', 'provider_id': 'ollama', 'provider_resource_id': 'granite4:latest'},
│   │   object='model'
│   ),
│   Model(
│   │   id='ollama/nomic-embed-text:latest',
│   │   created=1770492896,
│   │   owned_by='llama_stack',
│   │   custom_metadata={
│   │   │   'model_type': 'embedding',
│   │   │   'provider_id': 'ollama',
│   │   │   'provider_resource_id': 'nomic-embed-text:latest',
│   │   │   'embedding_dimension': 768,
│   │   │   'context_length': 8192
│   │   },
│   │   object='model'
│   ),
│   Model(
│   │   id='ollama/all-minilm:l6-v2',
│   │   created=1770492896,
│   │   owned_by='llama_stack',
│   │   custom_metadata={
│   │   │   'model_type': 'embedding',
│   │   │   'provider_id': 'ollama',
│   │   │   'provider_resource_id': 'all-minilm:l6-v2',
│   │   │   'embedding_dimension': 384,
│   │   │   'context_length': 512
│   │   },
│   │   object='model'
│   )
]

## Setup a simple RAG system using LangChain

NOTE: an alternative here is to use Llama Stack's client.vector_stores.create(), see: https://llamastack.github.io/docs/building_applications/rag, however that seems to work only with the files API, not the datasets API. Since our dataset is already in Hugging Face, the LangChain approach seems a bit more natural.

### Datasets we will use

- We will use `m-ric/huggingface_doc` as our knowledge base (the Hugging Face documentation).
- We will use `m-ric/huggingface_doc_qa_eval` as our ground truth for evaluation (questions and the expected answers).
- We will create `dmaniloff/hf_doc_qa_ragas_eval_dataset` as our evaluation dataset, which will be the result of using our RAG system to answer questions about the Hugging Face documentation.


In [7]:
# This is a datset of the Hugging Face documentation
dataset_name = "m-ric/huggingface_doc"
text_column = "text"

loader = HuggingFaceDatasetLoader(dataset_name, text_column)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
chunks = text_splitter.split_documents(documents)
print(f"Split blog post into {len(chunks)} sub-documents.")

INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/m-ric/huggingface_doc/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/m-ric/huggingface_doc/1b83935099b148190b6a9a9874b7e62a17fea889/README.md "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/m-ric/huggingface_doc/resolve/1b83935099b148190b6a9a9874b7e62a17fea889/huggingface_doc.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/m-ric/huggingface_doc/m-ric/huggingface_doc.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/m-ric/huggingface_doc/revision/1b83935099b148190b6a9a9874b7e62a17fea889 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/m-ric/huggingface_doc/resolve/1b83935099b148190b6a9a9874b7e62a17fea889/.huggingface.yaml "HTTP/1.1 404 Not Found"
INFO:httpx:

Split blog post into 29129 sub-documents.


In [8]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = InMemoryVectorStore(embeddings)
document_ids = vector_store.add_documents(documents=chunks[:10_000])  # make it quick
print(document_ids[:3])

/var/folders/8g/k40y0tk10_g28ck0h9nxph9m0000gn/T/ipykernel_59293/3100604079.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: mps
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c

['6331132e-ad82-44c6-b167-94e4e397936c', 'fe948d42-37a0-4b5c-9112-ee4ccaf0a713', 'fb4878c6-8551-4af8-b18e-208b8bfda09b']


### Simple RAG System Definition


In [9]:
rag_model = init_chat_model(model="granite3.3:2b", model_provider="ollama")
rag_model.invoke("Hello, world!")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={}, response_metadata={'model': 'granite3.3:2b', 'created_at': '2026-02-07T19:35:48.141242Z', 'message': {'role': 'assistant', 'content': ''}, 'done_reason': 'stop', 'done': True, 'total_duration': 1673196417, 'load_duration': 1466352709, 'prompt_eval_count': 47, 'prompt_eval_duration': 106682500, 'eval_count': 10, 'eval_duration': 98151625}, id='lc_run--019c399a-6820-7643-8aea-0dc2e3d1fee3-0', tool_calls=[], invalid_tool_calls=[])

In [10]:
def simple_rag_chain(
    model: BaseChatModel,
    vector_store: InMemoryVectorStore,
    query: str,
):
    """Inject retrieved documents into the system message."""
    retrieved_docs = vector_store.similarity_search(query)
    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)
    system_message = (
        "You are a helpful assistant. Use the following context in your response:"
        f"\n\n{docs_content}"
    )
    return {
        "answer": model.invoke(system_message).content,
        "sources": [d.metadata for d in retrieved_docs],
    }


In [11]:
# test it
simple_rag_chain(
    rag_model, vector_store, "How do I create a dataset card on Hugging Face?"
)

{'answer': "The Hugging Face Hub is a platform where datasets are shared and discovered by the community. To contribute to this repository, you should first ensure that your dataset follows best practices for creating a well-documented dataset card, as outlined in their guidelines. This information is primarily stored within each dataset repository's *README.md* file.\n\nTo begin, utilize the [`datasets-tagging` application](https://huggingface.co/datasets/tagging/), which helps create metadata tags in YAML format for search optimization and discoverability on the Hugging Face Hub. Clone this repository locally and run it to generate these tags.\n\nNext, refer to the [\\ud83e\\udd17 Datasets guide](https://github.com/huggingface/datasets/blob/master/templates/README_guide.md) for instructions on crafting informative dataset cards. A template can be found in the `lewtun/github-issues` dataset repository, which includes a filled-out *README.md* file as an example.\n\nOnce you've complete

## Evaluation Data Setup

### Prepare Evaluation Dataset
Here we prepare an evaluation dataset by asking the RAG system to answer questions
from the Hugging Face documentation and placing them side-by-side with the ground truth.

In [12]:
# This is a dataset of questions and answers using the Hugging Face documentation
# as a source.
ground_truth = load_dataset("m-ric/huggingface_doc_qa_eval", split="train")

evaluation_data = []
for row in tqdm(ground_truth.select(range(10))):
    question = row["question"]
    expected_answer = row["answer"]
    rag_result = simple_rag_chain(rag_model, vector_store, question)
    model_response = rag_result.get("answer", "")
    evaluation_data.append(
        {
            "user_input": question,  # ragas expects "user_input" as the key for the question
            "reference": expected_answer,  # ragas expects "reference" as the key for the ground truth answer
            "response": model_response,
        }
    )


INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/m-ric/huggingface_doc_qa_eval/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/m-ric/huggingface_doc_qa_eval/5f70aa9a1e2430f528ac3f27f01f0ba8719c0704/README.md "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/m-ric/huggingface_doc_qa_eval/resolve/5f70aa9a1e2430f528ac3f27f01f0ba8719c0704/huggingface_doc_qa_eval.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/m-ric/huggingface_doc_qa_eval/m-ric/huggingface_doc_qa_eval.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/m-ric/huggingface_doc_qa_eval/resolve/5f70aa9a1e2430f528ac3f27f01f0ba8719c0704/.huggingface.yaml "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://datasets-server.huggingface.co/info?dataset=m-ric/huggingface_doc_qa_eval

In [13]:
pprint(evaluation_data[:2])

[
│   {
│   │   'user_input': 'What architecture is the `tokenizers-linux-x64-musl` binary designed for?\n',
│   │   'reference': 'x86_64-unknown-linux-musl',
│   │   'response': "The provided information lists several binaries of the `tokenizers` package, each tailored for specific operating systems and architectures:\n\n1. `tokenizers-linux-arm64-musl`: This is a binary designed for AARCH64 (aarch64) systems using the musl C library on Linux. It's optimized for ARM architecture.\n2. `tokenizers-win32-x64-msvc`: This binary is built for x86_64 (also known as AMD64) systems utilizing the Microsoft Visual C++ (MSVC) compiler on Windows.\n3. `tokenizers-freebsd-x64`: This binary is compiled for x86_64 (amd64) architectures using the FreeBSD native toolchain, specifically targeting FreeBSD operating systems.\n4. `tokenizers-win32-ia32-msvc`: This binary is tailored for 32-bit x86 (i686) systems running on Windows, also built with the Microsoft Visual C++ (MSVC) compiler.\n\nThese binaries enable users to incorporate the `tokenizers` package into their respective environments, allowing them to utilize tokenization functionalities in their applications."
│   },
│   {
│   │   'user_input': 'What is the purpose of the BLIP-Diffusion model?\n',
│   │   'reference': 'The BLIP-Diffusion model is designed for controllable text-to-image generation and editing.',
│   │   'response': 'Diffusion models are machine learning systems trained to remove Gaussian noise progressively, aiming to generate samples of interest like images. They work by iteratively refining a random noise sample into a desired output, as detailed in this Colab: [diffusers/diffusers_intro.ipynb](https://colab.research.google.com/github/huggingface/notebooks/blob/main/diffusers/diffusers_intro.ipynb).\n\nOne major drawback of traditional diffusion models is the slow reverse denoising process due to their sequential nature, and they can be memory-intensive because they operate in pixel space, especially for high-resolution images. This complexity makes training and inference quite challenging.\n\nLatent diffusion addresses these issues by applying the diffusion process on a lower dimensional latent space rather than using the actual pixel space. This reduces both memory consumption and computational requirements. For more technical insights into how stable diffusion works, refer to our blog post: [Stable Diffusion with Annotated Diffusers](https://huggingface.co/blog/annotated-diffusion) or explore the original codebase at CompVis/stable-diffusion (GitHub) or Stability-AI/stablediffusion (GitHub).\n\nStable Diffusion v1.0 can be found at [CompVis/stable-diffusion](https://github.com/CompVis/stable-diffusion), and Stable Diffusion v2.0 at [Stability-AI/stablediffusion](https://github.com/Stability-AI/stablediffusion). Both contain original scripts for various tasks, along with official checkpoints on Hugging Face (CompVis), Runway, and Stability AI Hubs. You can explore these organizations to find the best checkpoint for your specific use case.\n\nCitation: [patil2022stable](https://doi.org/10.48550/arXiv.2209.14632) - "Stable Diffusion with Annotated Diffusers" by Suraj Patil, Pedro Cuenca, Nathan Lambert, and Patrick von Platen.'
│   }
]

In [14]:
ragas_eval_dataset = Dataset.from_list(evaluation_data)
ragas_eval_dataset.push_to_hub("dmaniloff/hf_doc_qa_ragas_eval_dataset")

INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/dmaniloff/hf_doc_qa_ragas_eval_dataset "HTTP/1.1 200 OK"
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 1210.48ba/s]
INFO:httpx:HTTP Request: POST https://huggingface.co/api/datasets/dmaniloff/hf_doc_qa_ragas_eval_dataset/preupload/main "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://huggingface.co/datasets/dmaniloff/hf_doc_qa_ragas_eval_dataset.git/info/lfs/objects/batch "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/dmaniloff/hf_doc_qa_ragas_eval_dataset/xet-write-token/main "HTTP/1.1 200 OK"
Processing Files (1 / 1): 100%|██████████| 18.8kB / 18.8kB, 94.0kB/s  
New Data Upload: 100%|██████████| 18.8kB / 18.8kB, 94.0kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.15 shards/s]
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/dmaniloff/hf_doc_qa_ragas_eval_dataset "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET ht

CommitInfo(commit_url='https://huggingface.co/datasets/dmaniloff/hf_doc_qa_ragas_eval_dataset/commit/f2a4e78541eda1cc09c77df0dec540dc08335c18', commit_message='Upload dataset', commit_description='', oid='f2a4e78541eda1cc09c77df0dec540dc08335c18', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/dmaniloff/hf_doc_qa_ragas_eval_dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='dmaniloff/hf_doc_qa_ragas_eval_dataset'), pr_revision=None, pr_num=None)

### Use the evaluation dataset in our Llama Stack distribution

Add to your stack config's registered_resources.datasets section:

```yaml
  registered_resources:
    datasets:
    - purpose: eval/messages-answer
      source:
        type: uri
        uri: huggingface://datasets/your-org/your-dataset?split=train
      metadata: {}
      dataset_id: your-dataset-id
```

And also include the huggingface datasetio provider:

```yaml
  providers:
    datasetio:
    - provider_id: huggingface
      provider_type: remote::huggingface
      config:
        kvstore:
          namespace: datasetio::huggingface
          backend: kv_default
```


In [15]:
# make sure the dataset is registered, you should see a uri like "huggingface://datasets/dmaniloff/hf_doc_qa_ragas_eval_dataset?split=train"
datasets = client.beta.datasets.list()
pprint(datasets)

INFO:httpx:HTTP Request: GET https://2b82-174-95-180-98.ngrok-free.app/v1beta/datasets "HTTP/1.1 200 OK"


[
│   DatasetListResponseItem(
│   │   identifier='hf_doc_qa_ragas_eval',
│   │   provider_id='huggingface',
│   │   purpose='eval/messages-answer',
│   │   source=DatasetListResponseItemSourceUriDataSource(
│   │   │   uri='huggingface://datasets/dmaniloff/hf_doc_qa_ragas_eval_dataset?split=train',
│   │   │   type='uri'
│   │   ),
│   │   metadata={},
│   │   provider_resource_id='hf_doc_qa_ragas_eval',
│   │   type='dataset'
│   )
]

### Benchmark Registration

Similarly, we will register a benchmark that defines what metrics to use for evaluation.

In your stack config's registered_resources.benchmarks section, add one benchmark for each provider you want to run. Notice in this case we will run both the inline and remote providers using the same dataset.

```yaml
   benchmarks:
    - benchmark_id: hf-doc-qa-ragas-inline-benchmark
      dataset_id: hf_doc_qa_ragas_eval
      scoring_functions:
        - semantic_similarity
      provider_id: trustyai_ragas_inline
      metadata: {}
    - benchmark_id: hf-doc-qa-ragas-remote-benchmark
      dataset_id: hf_doc_qa_ragas_eval
      scoring_functions:
        - semantic_similarity
      provider_id: trustyai_ragas_remote
      metadata: {}
```

In [16]:
# make sure the benchmarks are registered, you should see two benchmark ids: "hf-doc-qa-ragas-inline-benchmark" and "hf-doc-qa-ragas-remote-benchmark"
benchmarks = client.alpha.benchmarks.list()
pprint(benchmarks)

INFO:httpx:HTTP Request: GET https://2b82-174-95-180-98.ngrok-free.app/v1alpha/eval/benchmarks "HTTP/1.1 200 OK"


[
│   Benchmark(
│   │   dataset_id='hf_doc_qa_ragas_eval',
│   │   identifier='hf-doc-qa-ragas-inline-benchmark',
│   │   provider_id='trustyai_ragas_inline',
│   │   scoring_functions=['semantic_similarity'],
│   │   metadata={},
│   │   provider_resource_id='hf-doc-qa-ragas-inline-benchmark',
│   │   type='benchmark'
│   ),
│   Benchmark(
│   │   dataset_id='hf_doc_qa_ragas_eval',
│   │   identifier='hf-doc-qa-ragas-remote-benchmark',
│   │   provider_id='trustyai_ragas_remote',
│   │   scoring_functions=['semantic_similarity'],
│   │   metadata={},
│   │   provider_resource_id='hf-doc-qa-ragas-remote-benchmark',
│   │   type='benchmark'
│   )
]

In [17]:
# Find the benchmark identifiers for "inline" and "remote"
inline_benchmark_id = next(b.identifier for b in benchmarks if "inline" in b.identifier)
remote_benchmark_id = next(b.identifier for b in benchmarks if "remote" in b.identifier)

inline_benchmark_id, remote_benchmark_id

('hf-doc-qa-ragas-inline-benchmark', 'hf-doc-qa-ragas-remote-benchmark')

## Evaluation Execution

Run the evaluation using our Ragas out-of-tree provider.


In [25]:
# Review settings in distributinon/run.yaml, eg., note that
# since we can't set the embedding model in the benchmark config,
# the embedding model is set in the distribution run.yaml file(all-MiniLM-L6-v2)

remote_job = client.alpha.eval.run_eval(
    benchmark_id=remote_benchmark_id,
    benchmark_config={
        "eval_candidate": {
            "type": "model",
            "model": "ollama/granite4:latest",
            "sampling_params": {"temperature": 0.1, "max_tokens": 100},
        },
        "scoring_params": {},
        # "num_examples": 1,
    },
)
pprint(remote_job)

INFO:httpx:HTTP Request: POST https://2b82-174-95-180-98.ngrok-free.app/v1alpha/eval/benchmarks/hf-doc-qa-ragas-remote-benchmark/jobs "HTTP/1.1 200 OK"


Job(job_id='a6f27792-e1a7-45a4-8755-19b065febb17', status='in_progress')

In [26]:
# Review settings in distributinon/run.yaml, eg., note that
# since we can't set the embedding model in the benchmark config,
# the embedding model is set in the distribution run.yaml file(all-MiniLM-L6-v2)

inline_job = client.alpha.eval.run_eval(
    benchmark_id=inline_benchmark_id,  # this is the benchmark we registered above
    benchmark_config={
        "eval_candidate": {
            "type": "model",
            "model": "ollama/granite4:latest",  # this is the model we're using as a judge for the evaluation
            "sampling_params": {"temperature": 0.1, "max_tokens": 100},
        },
        "scoring_params": {},
        # "num_examples": 1,
    },
)
pprint(inline_job)

INFO:httpx:HTTP Request: POST https://2b82-174-95-180-98.ngrok-free.app/v1alpha/eval/benchmarks/hf-doc-qa-ragas-inline-benchmark/jobs "HTTP/1.1 200 OK"


Job(job_id='1', status='in_progress')

## Results Display


In [27]:
# wait a bit for the job to complete
pprint(
    client.alpha.eval.jobs.status(
        benchmark_id=inline_benchmark_id, job_id=inline_job.job_id
    )
)

INFO:httpx:HTTP Request: GET https://2b82-174-95-180-98.ngrok-free.app/v1alpha/eval/benchmarks/hf-doc-qa-ragas-inline-benchmark/jobs/1 "HTTP/1.1 200 OK"


Job(job_id='1', status='completed')

In [28]:
# wait a bit for the job to complete
pprint(
    client.alpha.eval.jobs.status(
        benchmark_id=remote_benchmark_id, job_id=remote_job.job_id
    )
)

INFO:httpx:HTTP Request: GET https://2b82-174-95-180-98.ngrok-free.app/v1alpha/eval/benchmarks/hf-doc-qa-ragas-remote-benchmark/jobs/a6f27792-e1a7-45a4-8755-19b065febb17 "HTTP/1.1 200 OK"


Job(job_id='a6f27792-e1a7-45a4-8755-19b065febb17', status='in_progress')

In [29]:
remote_results = client.alpha.eval.jobs.retrieve(
    benchmark_id=remote_benchmark_id, job_id=remote_job.job_id
)
pprint(remote_results)

INFO:httpx:HTTP Request: GET https://2b82-174-95-180-98.ngrok-free.app/v1alpha/eval/benchmarks/hf-doc-qa-ragas-remote-benchmark/jobs/a6f27792-e1a7-45a4-8755-19b065febb17/result "HTTP/1.1 200 OK"


EvaluateResponse(generations=[], scores={})

In [30]:
inline_results = client.alpha.eval.jobs.retrieve(
    benchmark_id=inline_benchmark_id, job_id=inline_job.job_id
)
pprint(inline_results)

INFO:httpx:HTTP Request: GET https://2b82-174-95-180-98.ngrok-free.app/v1alpha/eval/benchmarks/hf-doc-qa-ragas-inline-benchmark/jobs/1/result "HTTP/1.1 200 OK"


EvaluateResponse(
│   generations=[
│   │   {
│   │   │   'user_input': 'What architecture is the `tokenizers-linux-x64-musl` binary designed for?\n',
│   │   │   'response': "The provided information lists several binaries of the `tokenizers` package, each tailored for specific operating systems and architectures:\n\n1. `tokenizers-linux-arm64-musl`: This is a binary designed for AARCH64 (aarch64) systems using the musl C library on Linux. It's optimized for ARM architecture.\n2. `tokenizers-win32-x64-msvc`: This binary is built for x86_64 (also known as AMD64) systems utilizing the Microsoft Visual C++ (MSVC) compiler on Windows.\n3. `tokenizers-freebsd-x64`: This binary is compiled for x86_64 (amd64) architectures using the FreeBSD native toolchain, specifically targeting FreeBSD operating systems.\n4. `tokenizers-win32-ia32-msvc`: This binary is tailored for 32-bit x86 (i686) systems running on Windows, also built with the Microsoft Visual C++ (MSVC) compiler.\n\nThese binaries enable users to incorporate the `tokenizers` package into their respective environments, allowing them to utilize tokenization functionalities in their applications.",
│   │   │   'reference': 'x86_64-unknown-linux-musl'
│   │   },
│   │   {
│   │   │   'user_input': 'What is the purpose of the BLIP-Diffusion model?\n',
│   │   │   'response': 'Diffusion models are machine learning systems trained to remove Gaussian noise progressively, aiming to generate samples of interest like images. They work by iteratively refining a random noise sample into a desired output, as detailed in this Colab: [diffusers/diffusers_intro.ipynb](https://colab.research.google.com/github/huggingface/notebooks/blob/main/diffusers/diffusers_intro.ipynb).\n\nOne major drawback of traditional diffusion models is the slow reverse denoising process due to their sequential nature, and they can be memory-intensive because they operate in pixel space, especially for high-resolution images. This complexity makes training and inference quite challenging.\n\nLatent diffusion addresses these issues by applying the diffusion process on a lower dimensional latent space rather than using the actual pixel space. This reduces both memory consumption and computational requirements. For more technical insights into how stable diffusion works, refer to our blog post: [Stable Diffusion with Annotated Diffusers](https://huggingface.co/blog/annotated-diffusion) or explore the original codebase at CompVis/stable-diffusion (GitHub) or Stability-AI/stablediffusion (GitHub).\n\nStable Diffusion v1.0 can be found at [CompVis/stable-diffusion](https://github.com/CompVis/stable-diffusion), and Stable Diffusion v2.0 at [Stability-AI/stablediffusion](https://github.com/Stability-AI/stablediffusion). Both contain original scripts for various tasks, along with official checkpoints on Hugging Face (CompVis), Runway, and Stability AI Hubs. You can explore these organizations to find the best checkpoint for your specific use case.\n\nCitation: [patil2022stable](https://doi.org/10.48550/arXiv.2209.14632) - "Stable Diffusion with Annotated Diffusers" by Suraj Patil, Pedro Cuenca, Nathan Lambert, and Patrick von Platen.',
│   │   │   'reference': 'The BLIP-Diffusion model is designed for controllable text-to-image generation and editing.'
│   │   },
│   │   {
│   │   │   'user_input': 'How can a user claim authorship of a paper on the Hugging Face Hub?\n',
│   │   │   'response': 'The Hugging Face Hub is a powerful platform that facilitates the sharing and collaboration of machine learning models, datasets, and research within the natural language processing (NLP) community. It offers a wide array of features designed to promote ethical practices in AI development.\n\nFirstly, users can upload their custom datasets on the Hub for others to discover, use, and build upon. This open sharing of resources fosters collaboration and encourages responsible data handling practices. For detailed guidance on uploading data